# Observability Introduction: Start the DOG Stack

**Module 1: Observability Foundations in the DOG Stack** · [Quiz](quiz01_observability_foundations.ipynb) · [Course README](../../README.md)

Run the cells in this notebook in order. You start the DOG stack (Doris, the OpenTelemetry Collector, and Grafana), start the OpenTelemetry demo store, which keeps sending telemetry to the stack, check in Doris that the telemetry arrives, and then look at it in the storefront and in Grafana.

## 1. Start the DOG stack

The environment cell checks out a pinned version of the DOG stack under `.runtime/` and starts its four containers: `doris-obs` (Doris), `doris-obs-otel-collector`, `doris-obs-grafana`, and `doris-obs-mailpit`, a local mail server that receives Grafana's alert emails in Module 8. It also mounts the Doris log directories on this machine, so that Module 2 can load the FE log.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [ ]:
lab.prepare_environment()


**Expected result**

The host port check lists seven ports as `available`, or as `used by this course` when you rerun the cell. The command log ends with the four running containers. The Doris nodes table shows one FE and one BE, both alive, with the version `doris-4.1.3-rc02-7126cf65d96`, and the last card gives the Grafana, Doris, and Mailpit addresses. The first run downloads about 2.1 GB of images; a rerun reuses the running containers and takes a few seconds.

## 2. Start the OpenTelemetry demo

The OpenTelemetry demo is an online store, the Astronomy Shop, built from more than a dozen services that are instrumented with OpenTelemetry. Its load generator keeps browsing the store and placing orders, so the services keep producing traces, logs, and metrics. The demo's own Collector forwards them to the DOG Collector, which writes them to Doris. The later modules analyze this data.

The next cell starts the demo's 23 containers. The first run downloads about 2.7 GB of images, which takes a few minutes. The cell then waits until every container is running and the storefront answers.

In [ ]:
lab.start_demo()


**Expected result**

The port check shows `28082` as `available`, or as `used by this course` when you rerun the cell. The command log lists 23 containers named `doris-obs-demo-*`. The cell ends with the card "The demo is running": all 23 containers are running and the storefront answers, usually within half a minute. If a container restarted while the demo started, a table above the card lists it. It is running again, and the next check shows whether its telemetry arrives.

### Check that the telemetry reached Doris

Running containers do not show that any telemetry reached Doris. The DOG Collector writes each signal to its own table in the `otel` database: spans to `otel_traces`, logs to `otel_logs`, and metrics to one table per metric type, such as `otel_metrics_sum` for counters. The next cell counts, for each signal, the rows from the last five minutes and the services that sent them. The tables store timestamps in UTC, so the query compares them with `UTC_TIMESTAMP()`. The demo's services start one after another, so the cell reruns the query until all three signals have rows.

In [ ]:
demo_telemetry = lab.wait_for_rows(
    """
    SELECT signal, source_table, services, recent_rows, latest_timestamp
    FROM (
        SELECT 'traces' AS signal, 'otel_traces' AS source_table,
               COUNT(DISTINCT service_name) AS services, COUNT(*) AS recent_rows,
               MAX(timestamp) AS latest_timestamp
        FROM otel_traces
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'logs', 'otel_logs', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_logs
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'metrics', 'otel_metrics_sum', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_metrics_sum
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
    ) AS recent
    WHERE recent_rows > 0
    ORDER BY signal
    """,
    title="Telemetry from the last five minutes",
    hint="Read the demo Collector's log with docker logs doris-obs-demo-otel-collector and the DOG Collector's "
         "log with docker logs doris-obs-otel-collector, then rerun this cell.",
    min_rows=3,
    timeout_seconds=180,
)

**Expected result**

Three rows, `logs`, `metrics`, and `traces`, usually within a minute of starting the demo. `services` grows while the demo's services start: after a few minutes, about 18 services send traces (one of them, `frontend-web`, is the storefront code that runs in the browser), about 10 send logs, and about 20 send metrics. The metrics also include the Collectors' own metrics and the host and container metrics, which have an empty service name. `latest_timestamp` is within a few seconds of the current time in UTC.

## 3. Open the storefront

Visit [http://127.0.0.1:28082](http://127.0.0.1:28082) (or the port in `DORIS_OBS_DEMO_PORT`) to see the demo. Browse the products and add one to the cart: your requests produce telemetry next to the load generator's.

<img src="images/otel_demo.PNG" alt="OpenTelemetry ecommerce demo running in a browser" style="max-width:100%;height:auto;">

## 4. Open the DOG components

You can access the running DOG components through [http://127.0.0.1:23000/a/velodb-doris-app](http://127.0.0.1:23000/a/velodb-doris-app) (or the port in `DORIS_OBS_GRAFANA_PORT`). The environment cell in section 1 also prints this address. Grafana reads the demo's telemetry from Doris, and Module 3 explains its pages.

<img src="images/grafana_demo.PNG" alt="DOG components running in Grafana" style="max-width:100%;height:auto;">

## 5. Stop the environment

The demo runs 23 containers and keeps sending telemetry, which the later modules analyze, so leave it running while you work through the course. When you need the memory back, run `lab.stop_demo()`: the telemetry already in Doris stays, and `lab.start_demo()` starts the demo again.

When you finish the course, `lab.teardown()` removes the course's containers, networks, and volumes. See Stop and clean up in the [course README](../../README.md).

## Next

Open [Quiz 1: Observability Foundations in the DOG Stack](quiz01_observability_foundations.ipynb), then continue to [Module 2: Ingesting Observability Data](../module02-data-ingest/lab02_ingest_observability_data.ipynb).